In [ ]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from itertools import combinations
import warnings
import re

# Suppress specific warnings from statsmodels
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', RuntimeWarning)

def clean_name(name):
    """Cleans a column name for use in statsmodels formulas."""
    # Replace spaces and special characters with underscores
    name = re.sub(r'\s+', '_', str(name)) # Replace one or more spaces with underscore
    name = re.sub(r'[(),-/]+', '_', name) # Replace (),- / with underscore
    # Remove any trailing underscores that might result
    name = name.strip('_')
    # Ensure it's a valid Python identifier 
    # name = re.sub(r'\W|^(?=\d)', '_', name)
    return name

def identify_factors_and_metrics(df):
    """
    Identifies factor columns based on common categorical string values or low cardinality,
    and identifies numeric metric columns.
    """
    factor_columns = []
    metric_columns = []
    potential_factors = df.select_dtypes(include=['object', 'category']).columns.tolist()

    max_unique_for_factor = 5
    potential_numeric_factors = df.select_dtypes(include=['number']).columns.tolist()
    for col in potential_numeric_factors:
        if df[col].nunique(dropna=True) <= max_unique_for_factor:
            unique_vals_numeric = set(df[col].dropna().unique())
            if unique_vals_numeric.issubset({0, 1}) or unique_vals_numeric.issubset({-1, 1}):
                 potential_factors.append(col)


    known_factor_patterns = [
        {"Yes", "No"},
        {"BART", "BERT"},
        {"GPT", "Claude", "DeepSeek"}
    ]

    identified_factors = set()
    temp_df = df.copy() # Work on a copy for type checks/conversions initially

    for col in potential_factors:
        original_col_name = col # Keep original name for adding to set
        try:
            # Use original df for checking unique values
            unique_vals = set(df[original_col_name].dropna().astype(str).unique())
            is_known_factor = any(unique_vals == pattern for pattern in known_factor_patterns)
            is_low_cardinality_str = (df[original_col_name].dtype == 'object' or pd.api.types.is_categorical_dtype(df[original_col_name])) and df[original_col_name].nunique(dropna=True) <= max_unique_for_factor

            if is_known_factor or is_low_cardinality_str:
                identified_factors.add(original_col_name)
                # Ensure factor columns are treated as strings/objects for formula in the temporary df for checks
                if not pd.api.types.is_string_dtype(temp_df[original_col_name]) and not pd.api.types.is_categorical_dtype(temp_df[original_col_name]):
                     temp_df[original_col_name] = temp_df[original_col_name].astype(str)

        except Exception as e:
            print(f"Could not process column '{original_col_name}' for factor identification: {e}")
            continue

    factor_columns = list(identified_factors)

    all_numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    metric_columns = [col for col in all_numeric_cols if col not in factor_columns]

    final_metric_columns = []
    for col in metric_columns:
        try:
             # Check if conversion is possible on the original df
            pd.to_numeric(df[col])
            final_metric_columns.append(col)
        except (ValueError, TypeError): # Catch TypeError as well
            print(f"Warning: Column '{col}' identified as metric but contains non-numeric data. Skipping.")

    factor_columns = [f for f in factor_columns if f not in final_metric_columns]

    print(f"Original Factor Columns Identified: {factor_columns}")
    print(f"Original Metric Columns Identified: {final_metric_columns}")

    return factor_columns, final_metric_columns # Return original names

def clean_statsmodels_feature_name(feature_name):
    """Cleans up feature names from statsmodels output for final table."""
    name = feature_name.replace('C(', '').replace(')', '')
    name = name.replace(':', '_AND_')
    return name

def analyze_factorial_anova(csv_path, output_csv, max_interaction_order=2):
    """
    Performs factorial ANOVA using statsmodels and calculates percentage contributions.
    Cleans column names before analysis.

    Args:
        csv_path (str): Path to the input CSV file.
        output_csv (str): Path to save the results CSV file.
        max_interaction_order (int): Maximum order of interactions to include.
    """
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: Input file not found at {csv_path}")
        return None
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return None

    # Identify original factor and metric column names
    original_factor_names, original_metric_names = identify_factors_and_metrics(df)

    if not original_factor_names:
        print("Error: No factor columns identified. Cannot perform analysis.")
        return None
    if not original_metric_names:
        print("Error: No numeric metric columns identified. Cannot perform analysis.")
        return None

    # Create cleaned names and rename the DataFrame columns for the analysis
    all_original_names = original_factor_names + original_metric_names
    original_to_cleaned = {name: clean_name(name) for name in all_original_names}
    # Keep track of cleaned names
    cleaned_factor_names = [original_to_cleaned[name] for name in original_factor_names]
    cleaned_metric_names = [original_to_cleaned[name] for name in original_metric_names]

    df.rename(columns=original_to_cleaned, inplace=True)
    print(f"\nUsing cleaned factor names: {cleaned_factor_names}")
    print(f"Using cleaned metric names: {cleaned_metric_names}\n")

    results = []

    for metric in cleaned_metric_names: # Iterate through cleaned metric names
        print(f"Analyzing metric: {metric}")
        # Ensure metric column is numeric after renaming
        df[metric] = pd.to_numeric(df[metric], errors='coerce')
        analysis_df = df.dropna(subset=[metric] + cleaned_factor_names).copy() # Use cleaned names here

        if analysis_df.empty:
             print(f"Skipping metric '{metric}' due to missing values after dropping NaNs.")
             continue

        sufficient_variation = True
        for factor in cleaned_factor_names: # Use cleaned names here
            # Ensure factors are treated as categorical strings in the analysis subset
            analysis_df[factor] = analysis_df[factor].astype(str)
            if analysis_df[factor].nunique() < 2 :
                 print(f"Warning: Factor '{factor}' has less than 2 levels for metric '{metric}' after dropping NaNs. Skipping analysis for this metric.")
                 sufficient_variation = False
                 break
        if not sufficient_variation:
            continue

        # Build the formula string using cleaned names
        main_effects = [f"C({f})" for f in cleaned_factor_names]
        formula_terms = list(main_effects)

        for r in range(2, max_interaction_order + 1):
            for comb in combinations(main_effects, r):
                 formula_terms.append(':'.join(comb))

        # No backticks needed now as names are cleaned
        formula = f"{metric} ~ {' + '.join(formula_terms)}"
        print(f"Using formula: {formula}")

        try:
            model = smf.ols(formula, data=analysis_df).fit()
            anova_results = anova_lm(model, typ=2)
            ss_total = anova_results['sum_sq'].sum()

            if ss_total == 0 or pd.isna(ss_total): # Check for NaN total SS too
                print(f"Warning: Total Sum of Squares is zero or NaN for metric '{metric}'. Contributions cannot be calculated reliably.")
                continue

            # Map cleaned metric name back to original for the output table
            original_metric_name = next((k for k, v in original_to_cleaned.items() if v == metric), metric)

            for feature in anova_results.index:
                if feature == 'Residual':
                    continue

                ss_effect = anova_results.loc[feature, 'sum_sq']
                # Check for NaN SS effect
                if pd.isna(ss_effect):
                     contribution = 0 # Or handle as NaN if preferred
                     print(f"Warning: Sum of Squares for feature '{feature}' is NaN in metric '{metric}'. Contribution set to 0.")
                else:
                     contribution = (ss_effect / ss_total) * 100 if ss_total > 0 else 0


                # Clean the feature name from statsmodels output
                # (which used cleaned names like C(Case_study):C(LLM))
                cleaned_feature_output_name = clean_statsmodels_feature_name(feature)

                feature_for_table = cleaned_feature_output_name

                results.append({
                    "Feature": feature_for_table, # Use cleaned name from analysis
                    "Metric": original_metric_name, # Use original metric name for readability
                    "Contribution (%)": contribution,
                })

        except MemoryError:
             print(f"Error during model fitting for metric '{metric}': MemoryError. The dataset or model complexity might be too large.")
             continue # Skip to next metric
        except Exception as e:
            print(f"Error during model fitting or ANOVA for metric '{metric}': {e}")
            continue

    if not results:
        print("\nNo results were generated.")
        return None

    output_df = pd.DataFrame(results).pivot(
        index="Feature",
        columns="Metric",
        values="Contribution (%)"
    )
    output_df = output_df.fillna(0)

    # Ensure all potential features (based on cleaned names) are in the index
    cleaned_main_effects_terms = [clean_statsmodels_feature_name(f) for f in main_effects]
    all_feature_names_expected = set(cleaned_main_effects_terms)
    for r in range(2, max_interaction_order + 1):
        for comb in combinations(cleaned_main_effects_terms, r):
            all_feature_names_expected.add('_AND_'.join(sorted(comb))) # Ensure consistent ordering

    for feat in all_feature_names_expected:
         if feat not in output_df.index:
              output_df.loc[feat] = 0

    output_df = output_df.sort_index().reset_index().rename_axis(None, axis=1)

    try:
        output_df.to_csv(output_csv, index=False, float_format='%.2f')
        print(f"\nANOVA results saved to: {output_csv}")
    except Exception as e:
        print(f"Error saving results to CSV: {e}")
        return None

    return output_df

def main():
    """Main function to run the analysis."""
    input_file = "FinalResultsYesNo.csv"
    output_file = "anova_factorial_contributions.csv"
    max_interaction_order = 2

    analyze_factorial_anova(input_file, output_file, max_interaction_order)

if __name__ == "__main__":
    main()

Original Factor Columns Identified: ['Role', 'Case study', 'LLM', 'Summary', 'Insight', 'Example']
Original Metric Columns Identified: ['BLEU', 'METEOR', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Flesch Reading Ease', 'Coverage (Claude)', 'Faithfulness (GPT)']

Using cleaned factor names: ['Role', 'Case_study', 'LLM', 'Summary', 'Insight', 'Example']
Using cleaned metric names: ['BLEU', 'METEOR', 'ROUGE_1', 'ROUGE_2', 'ROUGE_L', 'Flesch_Reading_Ease', 'Coverage__Claude', 'Faithfulness__GPT']

Analyzing metric: BLEU
Using formula: BLEU ~ C(Role) + C(Case_study) + C(LLM) + C(Summary) + C(Insight) + C(Example) + C(Role):C(Case_study) + C(Role):C(LLM) + C(Role):C(Summary) + C(Role):C(Insight) + C(Role):C(Example) + C(Case_study):C(LLM) + C(Case_study):C(Summary) + C(Case_study):C(Insight) + C(Case_study):C(Example) + C(LLM):C(Summary) + C(LLM):C(Insight) + C(LLM):C(Example) + C(Summary):C(Insight) + C(Summary):C(Example) + C(Insight):C(Example)
Analyzing metric: METEOR
Using formula: METEOR ~ C(Ro